<a id="encrypted-machine-learning-pipeline"></a>
# Encrypted Machine Learning: Pipeline

This tutorial covers `src/concrete_fhe_toolkit/ml/pipeline.py`. `FHEPipeline` chains preprocessing transformers and a final model into a **single** FHE circuit. This means there is only one compile step, one key set, and intermediate values never leave the encrypted domain!

<a id="using-fhepipeline"></a>
## Using FHEPipeline

We can combine an `FHEBinner` (which groups continuous variables into categorical bins) with an `FHELogisticRegression` model.

In [ ]:
from concrete_fhe_toolkit.ml.classes import FHELogisticRegression
from concrete_fhe_toolkit.ml.pipeline import FHEPipeline
from concrete_fhe_toolkit.ml.preprocessing import FHEBinner

# Create the pipeline
# Feature 0 has bins: (-inf, 18), [18, 30), [30, 45), [45, 65), [65, inf)
# Feature 1 has bins: (-inf, 1000), [1000, 5000), [5000, 20000), [20000, inf)
pipeline = FHEPipeline([
    FHEBinner([[18, 30, 45, 65], [1000, 5000, 20000]]),
    FHELogisticRegression(weights=[3, 2], bias=-7),
])

# Compile the entire pipeline into one circuit
pipeline.compile(inputset=[[[20, 800], [70, 30000], [40, 4000]]], batch_size=1)

# Test: Age 35 (Bin 2), Salary 7000 (Bin 2)
# Logistic Regression sees [2, 2]
# Score = 3*2 + 2*2 - 7 = 6 + 4 - 7 = 3 >= 0 -> Class 1
pred = pipeline.predict([35, 7000])
assert pred == 1

print("✅ Encrypted FHEPipeline passed!")